# 01 · Linha de base supervisionada (embeddings + regressão logística)

Usa os embeddings congelados já extraídos (cache `.npy` do projeto `bilateral_leakage` ou os CSVs do PhysioNet).

| Cenário | Treino | Teste |
|---|---|---|
| `brset_crossfit` | BRSET graduável (com e sem diabetes), 5 partes por paciente | coorte BRSET (diabéticos) |
| `mbrset_crossfit` | mBRSET macular graduável (os dois olhos), 5 partes por paciente | coorte mBRSET |
| `brset_to_mbrset` | todo o BRSET | coorte mBRSET (validação externa entre câmeras) |

Referência anterior (`embeddings_experiments`, mBRSET macular, ICDR ≥ 2, sem edema): AUROC 0,880 (DINOv3 ConvNeXt-Tiny) e 0,855 (DINOv3 ViT-S/16).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import baseline, cohort, embeddings, metrics

In [ ]:
embeddings.available()

In [ ]:
cohorts = {ds: cohort.load(ds) for ds in ("brset", "mbrset")}
rows = []
for m in config.EMB_MODELS:
    try:
        pr = baseline.run(m, cohorts)
    except FileNotFoundError as e:
        print("pulando", m, "->", e)
        continue
    for (setting, ds), d in pr.groupby(["setting", "dataset"]):
        c = cohorts[ds].merge(d, on="image_id")
        c = c[c.split == "eval"]
        auc = metrics.auroc(c.referable, c.p, groups=c.patient_id)
        ap = metrics.auprc(c.referable, c.p, groups=c.patient_id)
        rows.append({"embedding": m, "cenario": setting, "n": len(c), "pacientes": c.patient_id.nunique(),
                     "positivos": int(c.referable.sum()),
                     "AUROC": metrics.fmt(*auc, pct=False), "AUPRC": metrics.fmt(*ap, pct=False), "_auc": auc[0]})
res = pd.DataFrame(rows).sort_values(["cenario", "_auc"], ascending=[True, False])
config.TABLES.mkdir(parents=True, exist_ok=True)
res.drop(columns="_auc").to_csv(config.TABLES / "baseline_auroc.csv", index=False)
res.drop(columns="_auc")

Escolha **um** embedding para a linha de base do artigo (registre a escolha antes de olhar os resultados dos VLMs).